In [ ]:
import os
import shutil
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

# ==========================================
# 1. ENVIRONMENT SETUP FOR KAGGLE
# ==========================================
print("Setting up Kaggle environment...")

# Define paths based on Kaggle's architecture
# NOTE: Change 'structured3d-stitched' and 'gan-code' to whatever you named your datasets!
DATASET_PATH = "/kaggle/input/structured3d-stitched/train"
CODE_INPUT_PATH = "/kaggle/input/gan-code/gan_part"
WORKING_DIR = "/kaggle/working/gan_part"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

# Copy the architecture and data_utils to the writable working directory
if not os.path.exists(WORKING_DIR):
    shutil.copytree(CODE_INPUT_PATH, WORKING_DIR)

# Append to sys.path so Python can import your modular files
sys.path.append(WORKING_DIR)

# Ensure checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Import your custom modules
from architecture.generator import Generator
from architecture.discriminator import Discriminator
from data_utils.dataloader import Pix2PixDataset

# ==========================================
# 2. HYPERPARAMETERS FOR CLOUD GPUs
# ==========================================
# Kaggle GPUs have 16GB VRAM, so we can increase batch size to speed up training
BATCH_SIZE = 8       # Increased from 1
NUM_WORKERS = 4      # Uses CPU cores to load images faster
LEARNING_RATE = 2e-4
EPOCHS = 100
L1_LAMBDA = 100

# ==========================================
# 3. CORE TRAINING LOOP
# ==========================================
def train_fn(disc, gen, loader, opt_disc, opt_gen, l1_loss, bce, scaler, device, epoch):
    loop = tqdm(loader, leave=True)

    for idx, (x, y) in enumerate(loop):
        x = x.to(device)
        y = y.to(device)

        # --- Train Discriminator ---
        with torch.cuda.amp.autocast(): # Mixed precision for massive speed boost on T4 GPUs
            y_fake = gen(x)
            D_real = disc(x, y)
            D_fake = disc(x, y_fake.detach())
            D_real_loss = bce(D_real, torch.ones_like(D_real))
            D_fake_loss = bce(D_fake, torch.zeros_like(D_fake))
            D_loss = (D_real_loss + D_fake_loss) / 2

        opt_disc.zero_grad()
        scaler.scale(D_loss).backward()
        scaler.step(opt_disc)

        # --- Train Generator ---
        with torch.cuda.amp.autocast():
            D_fake_preds = disc(x, y_fake)
            G_fake_loss = bce(D_fake_preds, torch.ones_like(D_fake_preds))
            G_L1_loss = l1_loss(y_fake, y) * L1_LAMBDA
            G_loss = G_fake_loss + G_L1_loss

        opt_gen.zero_grad()
        scaler.scale(G_loss).backward()
        scaler.step(opt_gen)
        scaler.update()

        loop.set_description(f"Epoch [{epoch}/{EPOCHS}]")
        loop.set_postfix(D_loss=D_loss.item(), G_loss=G_loss.item())

# ==========================================
# 4. EXECUTION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training initiated on device: {device}")

# Initialize networks and load to GPU
disc = Discriminator().to(device)
gen = Generator().to(device)

opt_disc = optim.Adam(disc.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))

bce = nn.BCEWithLogitsLoss()
l1_loss = nn.L1Loss()
scaler = torch.cuda.amp.GradScaler() # Handles mixed-precision scaling to prevent underflow

# Load Data with Kaggle optimizations (pin_memory and num_workers)
dataset = Pix2PixDataset(root_dir=DATASET_PATH)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Found {len(dataset)} images. Starting training...")

for epoch in range(1, EPOCHS + 1):
    train_fn(disc, gen, loader, opt_disc, opt_gen, l1_loss, bce, scaler, device, epoch)

    # Save checkpoints
    if epoch % 10 == 0 or epoch == EPOCHS:
        torch.save(gen.state_dict(), os.path.join(CHECKPOINT_DIR, f"generator_epoch_{epoch}.pth"))
        torch.save(disc.state_dict(), os.path.join(CHECKPOINT_DIR, f"discriminator_epoch_{epoch}.pth"))
        print(f"Saved checkpoints for Epoch {epoch}")

# ==========================================
# 5. PACKAGING FOR DOWNLOAD
# ==========================================
print("Training complete! Zipping checkpoints for download...")
shutil.make_archive("/kaggle/working/trained_checkpoints", 'zip', CHECKPOINT_DIR)
print("Done. You can now download 'trained_checkpoints.zip' from the Kaggle Output tab.")